# Démonstration du Modèle LMTL (Zooplancton)

Ce notebook démontre l'utilisation du nouveau module `seapopym.lmtl` pour simuler la dynamique d'une population de zooplancton (biomasse et production structurée par âge).

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from datetime import timedelta
from functools import partial

from seapopym.controller import SimulationController, SimulationConfig
from seapopym.lmtl.configuration import LMTLParams
from seapopym.lmtl.core import (
    compute_day_length,
    compute_mean_temperature,
    compute_recruitment_age,
    compute_production_initialization,
    compute_aging_tendency,
    compute_recruitment_tendency,
    compute_mortality_tendency,
)
from seapopym.standard.coordinates import Coordinates

## 1. Configuration

In [ ]:
# Configuration de la simulation
config = SimulationConfig(
    start_date="2020-01-01",
    end_date="2020-01-20", # 20 jours
    timestep=timedelta(days=1)
)

# Paramètres du zooplancton
zooplankton_params = LMTLParams(
    day_layer=100.0,
    night_layer=10.0,
    tau_r_0=5.0, # Recrutement à 5 jours à T_ref
    gamma_tau_r=0.0, # Pas d'effet thermique sur l'âge pour simplifier la démo
    E=0.5,
    lambda_0=0.05,
    gamma_lambda=0.0,
    T_ref=20.0
)

## 2. Création de l'Environnement (Forçages)

In [ ]:
times = np.arange(config.start_date, config.end_date + config.timestep, config.timestep, dtype="datetime64[ns]")
lats = np.array([0.0]) # Equateur
lons = np.array([0.0])
depths = np.array([10.0, 100.0])
cohorts = np.arange(0, 10) # 10 cohortes (jours)

# Température : 20°C partout pour simplifier
temp_data = np.ones((len(times), len(lats), len(lons), len(depths))) * 20.0

temperature = xr.DataArray(
    temp_data,
    coords={
        Coordinates.T: times,
        Coordinates.Y: lats,
        Coordinates.X: lons,
        Coordinates.Z: depths
    },
    dims=(Coordinates.T, Coordinates.Y, Coordinates.X, Coordinates.Z),
    name="temperature"
)

# Production Primaire (NPP) : Constante = 1.0
npp = xr.DataArray(
    np.ones((len(times), len(lats), len(lons))) * 1.0,
    coords={
        Coordinates.T: times,
        Coordinates.Y: lats,
        Coordinates.X: lons
    },
    dims=(Coordinates.T, Coordinates.Y, Coordinates.X),
    name="primary_production"
)

forcings = xr.Dataset({"temperature": temperature, "primary_production": npp})

## 3. État Initial

In [ ]:
# Biomasse initiale nulle
biomass_init = xr.DataArray(
    np.zeros((len(lats), len(lons))),
    coords={Coordinates.Y: lats, Coordinates.X: lons},
    dims=(Coordinates.Y, Coordinates.X),
    name="biomass"
)

# Production initiale nulle
production_init = xr.DataArray(
    np.zeros((len(lats), len(lons), len(cohorts))),
    coords={Coordinates.Y: lats, Coordinates.X: lons, "cohort": cohorts},
    dims=(Coordinates.Y, Coordinates.X, "cohort"),
    name="production"
)

# Variables système (dt, cohort_ages)
# dt en jours (float)
dt_val = config.timestep.total_seconds() / 86400.0

# On les ajoute au dataset initial pour qu'ils soient disponibles pour les fonctions
initial_state = xr.Dataset(
    {
        "biomass": biomass_init,
        "production": production_init,
        "dt": dt_val,
        "cohort_ages": xr.DataArray(cohorts, coords={"cohort": cohorts}, dims="cohort")
    }
)

## 4. Construction du Modèle (Blueprint)

In [ ]:
def configure_model(bp):
    bp.register_forcing("temperature", dims=(Coordinates.T, Coordinates.Y, Coordinates.X, Coordinates.Z))
    bp.register_forcing("primary_production", dims=(Coordinates.T, Coordinates.Y, Coordinates.X))
    
    # Enregistrement des variables système comme forcings/data nodes
    # Elles sont présentes dans l'état initial
    bp.register_forcing("dt")
    bp.register_forcing("cohort_ages")
    
    # Enregistrement des coordonnées utilisées comme inputs
    bp.register_forcing(Coordinates.T)
    bp.register_forcing(Coordinates.Y)
    
    bp.register_group(
        group_prefix="zoo",
        parameters=zooplankton_params,
        units=[
            {
                "func": compute_day_length,
                "output_mapping": {"output": "day_length"},
                "input_mapping": {"latitude": Coordinates.Y, "time": Coordinates.T}
            },
            {
                "func": compute_mean_temperature,
                "output_mapping": {"output": "mean_temperature"},
            },
            {
                "func": compute_recruitment_age,
                "output_mapping": {"output": "recruitment_age"},
            },
            {
                "func": compute_production_initialization,
                "output_mapping": {"production_init": "production_source_cohort_0"},
                "output_tendencies": {"production_init": "production"},
            },
            {
                "func": compute_aging_tendency,
                "output_mapping": {"aging_flux": "aging_tendency"},
                "output_tendencies": {"aging_flux": "production"},
            },
            {
                "func": compute_recruitment_tendency,
                "output_mapping": {
                    "recruitment_sink": "recruitment_sink",
                    "recruitment_source": "recruitment_source"
                },
                "output_tendencies": {
                    "recruitment_sink": "production",
                    "recruitment_source": "biomass"
                },
            },
            {
                "func": compute_mortality_tendency,
                "output_mapping": {"mortality_loss": "mortality_tendency"},
                "output_tendencies": {"mortality_loss": "biomass"},
            }
        ]
    )

## 5. Exécution

In [ ]:
controller = SimulationController(config, backend="sequential")
controller.setup(configure_model, initial_state, forcings)
controller.run()

## 6. Visualisation

In [ ]:
# On s'attend à voir :
# 1. La production augmenter à la cohorte 0 (source NPP)
# 2. Cette production se déplacer vers les cohortes suivantes (aging)
# 3. À l'âge 5 (recruitment_age), la production disparaître et la biomasse augmenter.

state = controller.state

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))

# Plot Production par cohorte (somme spatiale)
prod_ts = state["production"].sum(dim=[Coordinates.Y, Coordinates.X])
prod_ts.plot(ax=ax1, hue="cohort")
ax1.set_title("Production par Cohorte")

# Plot Biomasse
biomass_ts = state["biomass"].sum(dim=[Coordinates.Y, Coordinates.X])
biomass_ts.plot(ax=ax2)
ax2.set_title("Biomasse Totale")

plt.tight_layout()
plt.show()